In [50]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
from torch.utils.data import DataLoader
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
import pandas as pd
import os
import matplotlib.pyplot as plt
from torch.utils.data import Subset

device = 'mps' if torch.mps.is_available() else 'cpu'

In [51]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64):
        super().__init__()
        
        self.enc1 = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.enc2 = nn.Conv2d(base_channels, base_channels*2, 3, padding=1)
        self.enc3 = nn.Conv2d(base_channels*2, base_channels*4, 3, padding=1)
        
        self.dec2 = nn.Conv2d(base_channels*4, base_channels*2, 3, padding=1)
        self.dec1 = nn.Conv2d(base_channels*2, base_channels, 3, padding=1)
        self.out = nn.Conv2d(base_channels, 1, 1)

        self.proj2 = nn.Conv2d(base_channels*2, base_channels*4, 1)  # 추가
        self.proj1 = nn.Conv2d(base_channels, base_channels*2, 1)    # 추가
        
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        # t_emb는 timestep embedding (broadcasting)
        x1 = self.act(self.enc1(x))
        x2 = self.act(self.enc2(F.avg_pool2d(x1, 2)))
        x3 = self.act(self.enc3(F.avg_pool2d(x2, 2)))
        
        x = F.interpolate(x3, scale_factor=2)
        x = self.act(self.dec2(x + self.proj2(x2)))  # 수정

        x = F.interpolate(x, scale_factor=2)
        x = self.act(self.dec1(x + self.proj1(x1)))  # 수정
        
        out = self.out(x)
        return out

In [52]:
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi / 2) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

In [53]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # FFT 용으로 흑백 변환
    transforms.ToTensor()
])

In [54]:
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
small_dataset = Subset(dataset, range(5000))  # 0~4999번 이미지만 사용
dataloader = DataLoader(small_dataset, batch_size=32, shuffle=False)


In [56]:
# 모델
model = SimpleUNet(in_channels=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Noise scheduler
betas = cosine_beta_schedule(1000)
betas = torch.tensor(betas, device=device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def q_sample(x_start, t, noise):
    sqrt_alpha_cumprod = torch.sqrt(alphas_cumprod[t]).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
    sqrt_one_minus_alpha = torch.sqrt(1 - alphas_cumprod[t]).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
    return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha * noise

def p_losses(x_start, t):
    noise = torch.randn_like(x_start)
    x_noisy = q_sample(x_start, t, noise)
    noise_pred = model(x_noisy, t)
    return F.mse_loss(noise_pred, noise)

# Training
epochs = 100
for epoch in tqdm(range(epochs)):
    for batch in dataloader:
        x, _ = batch
        x = x.to(device)
        t = torch.randint(0, 1000, (x.size(0),), device=device).long()
        
        loss = p_losses(x, t)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch}: Loss {loss.item():.4f}")


/var/folders/9j/qchxlv312nb6gp3_d2789xdw0000gn/T/ipykernel_51635/403623061.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  betas = torch.tensor(betas, device=device)
  1%|          | 1/100 [00:06<10:27,  6.34s/it]

Epoch 0: Loss 0.2488


  2%|▏         | 2/100 [00:12<10:00,  6.13s/it]

Epoch 1: Loss 0.2842


  3%|▎         | 3/100 [00:18<09:51,  6.09s/it]

...

Epoch 67: Loss 0.0915


 69%|██████▉   | 69/100 [09:34<11:06, 21.51s/it]

Epoch 68: Loss 0.0589


 69%|██████▉   | 69/100 [09:57<04:28,  8.66s/it]


KeyboardInterrupt: 